# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Msdff/FlyRankAiAssignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
import os
os.chdir("..")
print(os.getcwd())

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


df["stale_bucket"] = df["days_since_last_update"].apply(
    lambda x: "stale (180+ days)" if x >= 180 else "fresh (<180 days)")
signal1 = df.groupby("stale_bucket").agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean())
).round(3)
print(signal1)

                       n  decline_rate
stale_bucket                          
fresh (<180 days)  29826         0.542
stale (180+ days)    174         0.471


**Signal 1:- Staleness.** 
Stale pages (180+ days) show a lower decline rate than fresh pages (54.2%, n=29,826), our result are opposite from our expectation . However, the stale group is very small (only 174 out of 30,000 pages), so this result should be treated with caution rather than as strong evidence. It shows that being old alone does not mean a page is declining. Old pages and new pages decline at almost the same rate. So we should not trust "staleness" by itself as a strong sign. We need to mix it with other clues too not use it alone.

In [9]:
df["visibility_bucket"] = df["impressions_90d"].apply(
    lambda x: "high visibility (500+)" if x >= 500 else "low visibility (<500)")
signal2 = df.groupby("visibility_bucket").agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean())
).round(3)
print(signal2)

                            n  decline_rate
visibility_bucket                          
high visibility (500+)  16726         0.596
low visibility (<500)   13274         0.475


Signal 2 :— Visibility.
We checked pages with lots of visitors (500+ impressions) against pages with few visitors. High-visibility pages show a decline rate of 59.6%, while low-visibility pages show 47.5%. So high-visibility pages decline more often, and this gap is real, not random, since we checked it on a big, fair sample. This makes sense because a page with a lot of visitors makes it easier to notice when traffic goes down, while a page with almost no visitors doesn't have much traffic to lose, so a decline is harder to see.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

df["reason_code"] = "stale_visible_page"
df["action"] = "review_for_refresh"

queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "client_id", "baseline_score", "reason_code", "action",
     "days_since_last_update", "impressions_90d", "avg_position", "trend_direction"]
]

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved", len(queue), "rows to work/outputs/baseline_action_score.csv")
queue.head(20)

Saved 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,avg_position,trend_direction
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,review_for_refresh,194,61678,19.7,down
16514,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,review_for_refresh,194,59472,24.8,down
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,review_for_refresh,194,25715,22.2,down
21268,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,review_for_refresh,193,13299,10.5,down
11489,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,review_for_refresh,194,7812,39.0,down
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,review_for_refresh,193,7558,17.9,down
698,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,review_for_refresh,194,4590,31.0,down
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,review_for_refresh,194,4556,16.4,down
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,review_for_refresh,194,4429,25.3,down
20837,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,review_for_refresh,193,1697,15.8,down


**Top-20 Review:-Observations:**

The top 17 rows are genuine matches for the rule: each has (days_since_last_update) above 180 and substantial visibility (impressions_90d in the thousands), producing meaningfully high scores. A notable pattern is that most of these top picks belong to a single client (client_7f2253d7e2)suggesting this client's content may be more broadly under-maintained than others, worth flagging as a client-level observation, not just a page-level one.

However, the last three rows (scores of 0) reveal a labeling issue: these pages have (days_since_last_update) = 20, meaning they are clearly NOT stale, yet they still carry the reason_code stale_visible_page. The score itself is correctly 0 (the rule's math worked as intended), but the reason_code column was applied uniformly to every row instead of only to rows that actually earned a nonzero score. This is a real limitation of the current implementation, the reason_code should be conditional on the score being greater than 0, and this will be fixed before the rule is reused for further work.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue.head(20)

,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,avg_position,trend_direction
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,review_for_refresh,194,61678,19.7,down
16514,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,review_for_refresh,194,59472,24.8,down
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,review_for_refresh,194,25715,22.2,down
21268,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,review_for_refresh,193,13299,10.5,down
11489,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,review_for_refresh,194,7812,39.0,down
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,review_for_refresh,193,7558,17.9,down
698,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,review_for_refresh,194,4590,31.0,down
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,review_for_refresh,194,4556,16.4,down
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,review_for_refresh,194,4429,25.3,down
20837,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,review_for_refresh,193,1697,15.8,down


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:**

The clearest weak picks are the last three rows in the top-20 output (content_c87291853cab,content_3dc420aa9809, content_6f2f3043b633). Each has (days_since_last_update) = 20, these pages were updated very recently. They are not stale at all. Their (baseline_score) is correctly 0, showing the rule's math did not actually flag them as priorities. However, they still carry the reason_code     (stale_visible_page), which is misleading: the reason_code column was applied to every row uniformly instead of being conditional on the score being greater than zero. This is a labeling bug in the current implementation, not a data issue, and it should be fixed so that reason codes only appear on rows the rule genuinely flagged.

A second, softer weak pattern:- most of the top 20 rows belong to a single client (client_7f2253d7e2). This may reflect a real pattern, it may simply mean this client has more total content in the dataset, inflating their presence at the top by volume alone. This is worth investigating further before treating it as a strong finding, right now it is only an observed pattern, not a confirmed one.

**Leakage check:**

The rule was built using only days_since_last_update and impressions_90d,  both real, observed signals that were known at the time of scoring, not values calculated from a future time window. No FlyRank product flags (such as health_score, priority_score, or action_type) were used anywhere in the scoring logic, since these were never part of the dataset to begin with. The trend_direction column was used only during the earlier signal verification step , to check whether staleness and visibility related to decline,it was never used as an input to the baseline_scor itself. No client names, URLs, or private queries appear anywhere in the output. The rule is therefore safe from both product-flag leakage and future-window leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.